In [0]:
# Databricks notebook source

from __future__ import annotations

import uuid

# ============================================================
# Widgets
# ============================================================
dbutils.widgets.text("catalog", "phc_txdot")
catalog = dbutils.widgets.get("catalog").strip()

# ============================================================
# Constants
# ============================================================
SCRIPT_NAME = "458_build_gold_vw_fact_project_item_bid.py"
RUN_ID = str(uuid.uuid4())

SOURCE_TABLE = f"{catalog}.gold.fact_project_item_bid"
TARGET_VIEW = f"{catalog}.gold.vw_fact_project_item_bid"
RUN_LOG_TABLE = f"{catalog}.staging.pipeline_run_log"

# ============================================================
# Helpers
# ============================================================
def sql_escape(value: str) -> str:
    return value.replace("'", "''")


def log_run(status: str, row_count: int | None, message: str) -> None:
    row_count_sql = "NULL" if row_count is None else str(row_count)
    spark.sql(f"""
        INSERT INTO {RUN_LOG_TABLE}
        VALUES (
              '{RUN_ID}'
            , '{SCRIPT_NAME}'
            , '{sql_escape(status)}'
            , {row_count_sql}
            , '{sql_escape(message)}'
            , current_timestamp()
        )
    """)

# ============================================================
# Build gold.vw_fact_project_item_bid
# ============================================================
try:
    spark.sql(f"DROP VIEW IF EXISTS {TARGET_VIEW}")

    spark.sql(f"""
    CREATE VIEW {TARGET_VIEW} AS
    SELECT
          bid_item_fact_key                           AS `Bid Item Fact Key`
        , project_key                                 AS `Project Key`
        , vendor_key                                  AS `Vendor Key`
        , item_specification_key                      AS `Item Specification Key`

        , project_classification_key                  AS `Project Classification Key`
        , county_key                                  AS `County Key`

        , CASE
              WHEN specification_code IS NOT NULL
               AND specification_description IS NOT NULL
              THEN concat(CAST(specification_code AS STRING), ' - ', specification_description)
              WHEN specification_code IS NOT NULL
              THEN CAST(specification_code AS STRING)
              ELSE specification_description
          END                                         AS `Specification`

        , md5(COALESCE(item_work_category, ''))       AS `Item Work Category Key`

        , project_id                                  AS `Project ID`
        , vendor_name                                 AS `Vendor Name`
        , vendor_name_standardized                    AS `Vendor Name Standardized`
        , project_name                                AS `Project Name`
        , project_number                              AS `Project Number`
        , state_project_number                        AS `State Project Number`
        , control_section_job_csj                     AS `Control Section Job CSJ`
        , controlling_project_id_ccsj                 AS `Controlling Project ID CCSJ`
        , project_type                                AS `Project Type`
        , project_classification                      AS `Project Classification`
        , CAST(project_actual_let_date AS DATE)       AS `Project Actual Let Date`
        , project_estimated_let_date                  AS `Project Estimated Let Date`
        , county                                      AS `County`
        , location                                    AS `Location`
        , district_division                           AS `District Division`
        , highway                                     AS `Highway`
        , short_description                           AS `Short Description`

        , bid_submit_sequence_number                  AS `Bid Submit Sequence Number`
        , bid_rank_sequence_number                    AS `Bid Rank Sequence Number`
        , low_bidder_flag                             AS `Low Bidder Flag`
        , low_bidder_flag_int                         AS `Low Bidder Flag Int`
        , bid_code                                    AS `Bid Code`
        , alternative_bid_code                        AS `Alternative Bid Code`
        , bid_item_sequence_number                    AS `Bid Item Sequence Number`
        , bid_item_description                        AS `Bid Item Description`
        , measurement_unit                            AS `Measurement Unit`
        , bid_item_quantity                           AS `Bid Item Quantity`
        , bid_item_unit_price_amount                  AS `Bid Item Unit Price Amount`
        , bid_total_amount                            AS `Bid Total Amount`
        , specification_code                          AS `Specification Code`
        , specification_description                   AS `Specification Description`
        , effective_specification_description         AS `Effective Specification Description`
        , item_work_category                          AS `Item Work Category`
        , item_work_category_source                   AS `Item Work Category Source`
        , is_defaulted_work_category                  AS `Is Defaulted Work Category`
        , extended_item_amount_calc                   AS `Extended Item Amount Calc`
        , is_low_bid_project_vendor                   AS `Is Low Bid Project Vendor`
        , project_bid_rank                            AS `Project Bid Rank`
        , bidder_count_in_project                     AS `Bidder Count In Project`
        , lowest_bid_amount_in_project                AS `Lowest Bid Amount In Project`
        , highest_bid_amount_in_project               AS `Highest Bid Amount In Project`
        , bid_spread_amount_in_project                AS `Bid Spread Amount In Project`
        , bid_vs_low_bid_ratio                        AS `Bid Vs Low Bid Ratio`
        , item_amount_as_pct_of_low_bid_project_total AS `Item Amount As Pct Of Low Bid Project Total`
        , item_amount_as_pct_of_vendor_bid_total      AS `Item Amount As Pct Of Vendor Bid Total`
        , vendor_project_has_resubmission_history     AS `Vendor Project Has Resubmission History`
        , item_has_resubmission_history               AS `Item Has Resubmission History`
        , item_changed_across_submit_sequences        AS `Item Changed Across Submit Sequences`
        , is_latest_submit_for_vendor_project         AS `Is Latest Submit For Vendor Project`
        , item_missing_from_latest_project_submit     AS `Item Missing From Latest Project Submit`
        , item_changed_value_across_submits           AS `Item Changed Value Across Submits`
        , is_payment_adjustment_item                  AS `Is Payment Adjustment Item`
        , is_special_deduction_item                   AS `Is Special Deduction Item`
        , is_nonstandard_adjustment_item              AS `Is Nonstandard Adjustment Item`
    FROM {SOURCE_TABLE}
    """)

    row_count = spark.sql(f"SELECT COUNT(*) AS row_count FROM {TARGET_VIEW}").collect()[0]["row_count"]

    log_run("SUCCESS", row_count, "Built gold.vw_fact_project_item_bid successfully.")

    print("=" * 70)
    print("Built gold.vw_fact_project_item_bid")
    print(f"Row count: {row_count:,}")
    print("=" * 70)

except Exception as exc:
    log_run("FAILED", None, str(exc))
    raise